# CEG-WM：保存图像的残余平移／尺度敏感性诊断

**不生成新图像，不加载去噪器、文本编码器、DINO 或 SyncSeal 模型。仅加载原版本 VAE 评分。**

沿用刚才成功的 **Python 3.12 GPU 环境**（L4 即可），Secrets 沿用 `HF_TOKEN`、`CEG_WM_ROOT_KEY`。
运行全部单元。默认只读输入为 `MyDrive/CEG-WM/crop-bottleneck-v1/attempt01/`，
新结果写到 `MyDrive/CEG-WM/crop-residual-v1/attempt01/`。输入缺失会停止，绝不回退到生成。

固定 8 对图像，每张 22 个评分条件，共 352 条。先通过全部 32 个旧 Oracle／预测 H 重放检查，才测扰动。
CPU 工程检查不代表真实 GPU 诊断已经通过；本 Notebook 将给出新结果。


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, sys, json, re, time
SOURCE = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-bottleneck-v1/attempt01')
RUN_ID = 'attempt01'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
output = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-residual-v1') / RUN_ID
checkout = pathlib.Path('/content/cegwm-crop-residual-source')
script = pathlib.Path('/content/run_crop_residual.py')
expected = 'd9ef4adb4f8424f1102f577bb76347fbba112bd9'
saved_env = json.loads((SOURCE / 'environment.json').read_text())
saved_protocol = json.loads((SOURCE / 'protocol.json').read_text())
saved_summary = json.loads((SOURCE / 'summary.json').read_text())
if saved_summary['status'] != 'COMPLETE_DIAGNOSTIC':
    raise RuntimeError('Original crop run is incomplete. No generation will be started.')
source_python = tuple(map(int, saved_env['python'].split('.')[:2]))
if sys.version_info[:2] != source_python:
    raise RuntimeError(f'Use the successful historical Colab Python {source_python[0]}.{source_python[1]} runtime; current Python is {sys.version.split()[0]}. No dependencies or source results were changed.')
child_env = dict(os.environ)
for name in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f'Missing secret: {name}')
    child_env[name] = value
child_env['HF_HOME'] = '/content/cegwm-crop-v1-hf'
child_env['HF_HUB_CACHE'] = child_env['HF_HOME'] + '/hub'
child_env['PYTHONUNBUFFERED'] = '1'
child_env.pop('HF_HUB_OFFLINE', None)
child_env.pop('TRANSFORMERS_OFFLINE', None)

def run_logged(command, label, **kwargs):
    # Explicit PIPE+print: Colab may hide output inherited by subprocess.run.
    log_dir = output / 'execution_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log = log_dir / f'{label}-{time.time_ns()}.log'
    with log.open('x', encoding='utf-8') as sink:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            for line in process.stdout:
                for key in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
                    line = line.replace(child_env[key], '[REDACTED]')
                print(line, end='', flush=True)
                sink.write(line)
                sink.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed with exit code {code}. Full output: {log}')

run_logged([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print({"python":__import__("sys").version,"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__})'], 'gpu-check')
print('Read-only source:', SOURCE)
print('New output:', output)


In [ ]:
# Match the successful source reader dependencies; preserve installed CUDA torch.
packages = saved_env['packages']
names = ('diffusers','transformers','accelerate','numpy','Pillow','huggingface_hub','safetensors','sentencepiece')
dependencies = [f'{name}=={packages[name]}' for name in names]
torch_versions = subprocess.run([sys.executable, '-c',
    'import importlib.metadata as m,json; print(json.dumps({p:m.version(p) for p in ("torch","torchvision")}))'],
    capture_output=True, text=True, check=True)
constraints = pathlib.Path('/content/cegwm-residual-cuda-constraints.txt')
constraints.write_text(''.join(f'{name}=={ver}\n' for name,ver in json.loads(torch_versions.stdout).items()))
run_logged([sys.executable, '-m', 'pip', 'install', '--only-binary=sentencepiece',
            '-c', str(constraints), *dependencies], 'pip-install')
if not checkout.exists():
    run_logged(['git', 'clone', 'https://github.com/RICHAAARC/CEG-WM.git', str(checkout)], 'clone')
run_logged(['git', '-C', str(checkout), 'checkout', '--detach', expected], 'checkout')
child_env['PYTHONPATH'] = str(checkout / 'src') + os.pathsep + str(checkout)
run_logged([sys.executable, '-c',
    'from diffusers import AutoencoderKL; from diffusers.image_processor import VaeImageProcessor; '
    'from cegwm.runtime.blind_scoring_v2 import _lf,_hf; '
    'print("VAE and frozen scoring imports OK; no models loaded")'], 'imports', env=child_env)


In [ ]:
# Self-contained residual-only script.
script.write_text('"""Read-only saved-image diagnostic. No generator, denoiser, or SyncSeal inference."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport uuid\n\nEXACT = "d9ef4adb4f8424f1102f577bb76347fbba112bd9"\nMODEL = "stabilityai/stable-diffusion-3.5-medium"\nTAU = 1.5545224858704942\nROLES = ("positive", "negative")\nIDS = tuple(f"v2-formal_evaluation_pairs-{i:04d}" for i in range(1, 9))\nATOL, RTOL = 1e-5, 1e-4\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for part in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with tmp.open("x", encoding="utf-8") as f:\n        json.dump(data, f, ensure_ascii=False, indent=2, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef bind(path, data):\n    if Path(path).exists():\n        if read(path) != data:\n            raise RuntimeError(f"Resume identity mismatch: {Path(path).name}; do not mix runs")\n    else:\n        write(path, data)\n\n\ndef error_data(error):\n    trace = "".join(traceback.format_exception(error))\n    for name in ("HF_TOKEN", "CEG_WM_ROOT_KEY"):\n        if os.environ.get(name):\n            trace = trace.replace(os.environ[name], "[REDACTED]")\n    return {"error_type": type(error).__name__, "traceback_redacted": trace}\n\n\ndef cases():\n    result = [{"id": "oracle_replay", "kind": "oracle"},\n              {"id": "predicted_replay", "kind": "predicted"},\n              {"id": "oracle_uncorrected_pixel_convention", "kind": "convention"},\n              {"id": "half_prediction_residual", "kind": "half_residual"}]\n    for axis in ("x", "y"):\n        for value in (-4., -2., -.5, .5, 2., 4.):\n            result.append({"id": f"translation_{axis}_{value:+g}px", "kind": "translation",\n                           "axis": axis, "value": value})\n    for value in (-.04, -.02, -.005, .005, .02, .04):\n        result.append({"id": f"scale_{value * 100:+g}pct", "kind": "scale", "value": value})\n    return result\n\n\ndef normalization():\n    import numpy as np\n    return np.array([[2 / 511, 0., -1.], [0., 2 / 511, -1.], [0., 0., 1.]])\n\n\ndef matrix_for(case, crop, predicted):\n    """All perturbations compose BEFORE one sampling from saved attacked RGB."""\n    import numpy as np\n    base = np.asarray(crop["sampler_h_observed_to_canonical"], dtype=float)\n    kind = case["kind"]\n    if kind == "oracle":\n        return base\n    if kind == "predicted":\n        return np.asarray(predicted, dtype=float)\n    if kind == "convention":\n        return np.asarray(crop["physical_h_observed_to_canonical"], dtype=float)\n    if kind == "half_residual":\n        # A single predeclared diagnostic point along the actual predicted-H error.\n        # No interpolation weight is selected using detection scores.\n        p = np.asarray(predicted, dtype=float)\n        return .5 * base / base[2, 2] + .5 * p / p[2, 2]\n    residual = np.eye(3)\n    if kind == "translation":\n        residual[0 if case["axis"] == "x" else 1, 2] = case["value"]\n    elif kind == "scale":\n        scale = 1 + case["value"]\n        residual[0, 0] = residual[1, 1] = scale\n        residual[0, 2] = residual[1, 2] = 256 * (1 - scale)\n    else:\n        raise ValueError("Unknown fixed diagnostic case")\n    n = normalization()\n    return n @ residual @ np.linalg.inv(n) @ base\n\n\ndef geometry_metrics(matrix, crop):\n    import numpy as np\n    n = normalization()\n    edge = np.linalg.inv(n) @ matrix @ n\n    plus = np.array([[1., 0., .5], [0., 1., .5], [0., 0., 1.]])\n    center = np.linalg.inv(plus) @ edge @ plus\n    actual = n @ center @ np.linalg.inv(n)\n    truth = np.asarray(crop["physical_h_observed_to_canonical"])\n    points = np.array([[-1., -1., 1.], [1., -1., 1.], [1., 1., 1.], [-1., 1., 1.], [0., 0., 1.]]).T\n    a, b = actual @ points, truth @ points\n    delta = (a[:2] / a[2] - b[:2] / b[2]) * 511 / 2\n    return {"effective_corner_rmse_px": float(np.sqrt(np.mean(np.sum(delta[:, :4] ** 2, axis=0)))),\n            "effective_center_shift_xy_px": delta[:, 4].tolist()}\n\n\ndef load_saved_image(source, artifact):\n    from PIL import Image\n    path = Path(artifact["path"]).resolve()\n    if not path.is_relative_to(source.resolve()):\n        raise RuntimeError("Saved image points outside SOURCE directory")\n    if sha(path) != artifact["sha256"]:\n        raise RuntimeError(f"Saved image hash differs: {path.name}")\n    with Image.open(path) as image:\n        if image.mode != "RGB" or image.size != (512, 512):\n            raise ValueError("Saved input must be RGB 512x512")\n        return image.copy()\n\n\ndef inspect_source(source):\n    protocol, summary = read(source / "protocol.json"), read(source / "summary.json")\n    if summary["status"] != "COMPLETE_DIAGNOSTIC" or protocol["source_exact"] != EXACT:\n        raise ValueError("Completed original crop diagnostic required")\n    if tuple(u["unit_id"] for u in protocol["roster"]) != IDS or protocol["reference_tau"] != TAU:\n        raise ValueError("Source roster or threshold differs")\n    indexed = {(r["unit_id"], r["role"], r["route"]): r for r in summary["records"]}\n    if len(indexed) != 64 or len(summary["records"]) != 64:\n        raise ValueError("Expected 64 unique source route records")\n    for uid in IDS:\n        for role in ROLES:\n            for route in ("clean", "crop_direct", "oracle_h", "predicted_h"):\n                row = indexed[uid, role, route]\n                if row["status"] != "SCORED":\n                    raise ValueError("This fixed diagnostic requires all saved baseline routes")\n                load_saved_image(source, row["image"])\n    return protocol, indexed\n\n\nclass Reader:\n    """Only the VAE plus the unchanged LF/HF arithmetic from the pinned source."""\n    def __init__(self, repo, source):\n        import torch\n        from diffusers import AutoencoderKL\n        from diffusers.image_processor import VaeImageProcessor\n        from cegwm.method.lf import (FrozenLFPublicAssets, LF_BALANCED_BLOCKS_CARRIER_METHOD_ID,\n            LF_BLOCKNORM_DETECTOR_STATISTIC_ID, LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        from cegwm.method.hf import FrozenHFPublicAssets\n        from cegwm.method.content_whitening import FrozenContentWhiteningLFPublicAssets\n        from experiments.run_blind_detection_v1 import load_whitening_asset_semantic, load_weighted_asset_semantic\n        from cegwm.shared.keys import normalize_detection_key\n        from cegwm.runtime.content_weighted_joint_sd35 import derive_stability_wrong_keys\n        revision = read(source / "model_revisions.json")[MODEL]\n        self.vae = AutoencoderKL.from_pretrained(MODEL, subfolder="vae", revision=revision,\n            torch_dtype=torch.float16, token=os.environ["HF_TOKEN"]).to("cuda").eval()\n        # Exactly StableDiffusion3Pipeline v0.39.0\'s image_processor construction.\n        self.processor = VaeImageProcessor(vae_scale_factor=2 ** (len(self.vae.config.block_out_channels) - 1))\n        lf = FrozenLFPublicAssets(self.vae, self.processor, MODEL + ":image_processor",\n            LF_BALANCED_BLOCKS_CARRIER_METHOD_ID, LF_BLOCKNORM_DETECTOR_STATISTIC_ID,\n            LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        self.lf = FrozenContentWhiteningLFPublicAssets(lf, load_whitening_asset_semantic(repo))\n        self.hf = FrozenHFPublicAssets(self.vae, self.processor, MODEL + ":image_processor")\n        self.weighted = load_weighted_asset_semantic(repo)\n        self.key = normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])\n        self.keys = (self.key, *derive_stability_wrong_keys(self.key))\n\n    def observe(self, image):\n        from cegwm.runtime.observation import encode_final_rgb_image\n        return encode_final_rgb_image(image, self.processor, self.vae)\n\n    def score(self, observation):\n        from cegwm.runtime.blind_scoring_v2 import _lf, _hf\n        from cegwm.method.content_weighted_joint import weighted_joint_score\n        from cegwm.method.blind_detection import statistic_from_weighted_scores\n        labels = ("registered", *(f"wrong_{i:02d}" for i in range(16)))\n        pairs = [(_lf(observation, key, self.lf), _hf(observation, key, self.hf)) for key in self.keys]\n        branches = {"lf": {label: p[0] for label, p in zip(labels, pairs)},\n                    "hf": {label: p[1] for label, p in zip(labels, pairs)},\n                    "weighted_joint": {label: weighted_joint_score(*p, self.weighted) for label, p in zip(labels, pairs)}}\n        stat = statistic_from_weighted_scores(branches["weighted_joint"])\n        return {"branches": branches, "statistic": asdict(stat), "margin": stat.value - TAU,\n                "above_reference_tau": stat.value > TAU}\n\n\ndef replay_comparison(current, old):\n    import numpy as np\n    diffs, passed = {}, True\n    for branch in ("lf", "hf", "weighted_joint"):\n        a = list(current["branches"][branch].values())\n        b = list(old["branches"][branch].values())\n        diffs[branch] = float(np.max(np.abs(np.asarray(a) - b)))\n        passed &= bool(np.allclose(a, b, atol=ATOL, rtol=RTOL))\n    passed &= bool(np.isclose(current["statistic"]["value"], old["statistic"]["value"], atol=ATOL, rtol=RTOL))\n    return {"passed": passed, "max_abs_branch_differences": diffs,\n            "statistic_difference": current["statistic"]["value"] - old["statistic"]["value"],\n            "atol": ATOL, "rtol": RTOL}\n\n\ndef observation_metrics(image, oracle_image, observation, oracle_observation):\n    import numpy as np\n    a, b = np.asarray(image, dtype=float) / 255, np.asarray(oracle_image, dtype=float) / 255\n    z, z0 = observation.detach().float().cpu().numpy(), oracle_observation.detach().float().cpu().numpy()\n    def relative(x, y):\n        return float(np.linalg.norm(x - y) / max(float(np.linalg.norm(y)), 1e-12))\n    return {"rgb_rmse_full": float(np.sqrt(np.mean((a-b)**2))),\n            "rgb_rmse_fixed_inner": float(np.sqrt(np.mean((a[64:448,64:448]-b[64:448,64:448])**2))),\n            "vae_relative_l2_full": relative(z, z0),\n            "vae_relative_l2_fixed_inner": relative(z[...,8:56,8:56], z0[...,8:56,8:56])}\n\n\ndef summarize(output):\n    rows, counts = [], {}\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases():\n                path = output / "records" / uid / role / (case["id"] + ".json")\n                row = read(path) if path.exists() else {"status": "PENDING"}\n                rows.append({**row, "unit_id": uid, "role": role, "case": case["id"]})\n    for role in ROLES:\n        for case in cases():\n            subset = [r for r in rows if r["role"] == role and r["case"] == case["id"]]\n            scored = [r for r in subset if r["status"] == "SCORED"]\n            counts[f"{role}/{case[\'id\']}"] = {"planned": 8, "scored": len(scored),\n                "above_reference_tau": sum(r.get("above_reference_tau", False) for r in scored),\n                "operational_error": sum(r["status"] == "OPERATIONAL_ERROR" for r in subset),\n                "pending": sum(r["status"] == "PENDING" for r in subset),\n                "mean_margin": sum(r["margin"] for r in scored) / len(scored) if scored else None}\n    result = {"status": "COMPLETE_DIAGNOSTIC" if all(r["status"] == "SCORED" for r in rows) else "INCOMPLETE",\n              "source_pairs": 8, "cases_per_image": len(cases()), "planned_scores": len(rows),\n              "claim_ceiling": "fixed_grid_sensitivity_diagnosis_not_blind_recovery_or_calibrated_FPR",\n              "counts": counts, "records": rows}\n    write(output / "summary.json", result)\n    cols = ["unit_id", "role", "case", "status", "margin", "delta_score_vs_oracle",\n            "delta_registered_vs_oracle", "delta_wrongmax_vs_oracle", "delta_hf_vs_oracle",\n            "delta_lf_vs_oracle", "oracle_above_reference_tau", "lost_vs_oracle", "effective_corner_rmse_px",\n            "rgb_rmse_full", "rgb_rmse_fixed_inner", "vae_relative_l2_full", "vae_relative_l2_fixed_inner"]\n    tmp = output / "readout.csv.tmp"\n    with tmp.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=cols)\n        writer.writeheader()\n        writer.writerows({k: r.get(k) for k in cols} for r in rows)\n    os.replace(tmp, output / "readout.csv")\n    return result\n\n\ndef run(source, output, repo):\n    import numpy as np\n    import torch\n    from cegwm.geometry_v7.r1b import rectify_attacked_rgb\n    from cegwm.shared.keys import normalize_detection_key, public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must be separate non-overlapping directories")\n    protocol, old = inspect_source(source)  # Read-only verification BEFORE any inference.\n    if subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() != EXACT:\n        raise ValueError("Historical source revision differs")\n    if subprocess.check_output(["git", "-C", str(repo), "diff", "HEAD", "--"], text=True):\n        raise ValueError("Tracked historical source has modifications")\n    if public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])) != protocol["key_digest"]:\n        raise ValueError("Detection key differs from saved run")\n    identity = {"source": str(source.resolve()), "source_protocol_sha": sha(source / "protocol.json"),\n                "source_summary_sha": sha(source / "summary.json"), "source_model_revisions_sha": sha(source / "model_revisions.json"),\n                "source_environment_sha": sha(source / "environment.json"), "script_sha": sha(__file__),\n                "code_exact": EXACT, "cases": cases(), "tau_reference_only": TAU, "atol": ATOL, "rtol": RTOL}\n    bind(output / "protocol.json", identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA VAE reader required; no image generation fallback")\n    env = {"python": platform.python_version(), "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,\n           "packages": {p: version(p) for p in ("torch", "torchvision", "diffusers", "transformers", "numpy", "Pillow", "huggingface_hub", "safetensors")}}\n    bind(output / "environment.json", env)\n    if summarize(output)["status"] == "COMPLETE_DIAGNOSTIC":\n        print("Already complete; no scoring repeated.", flush=True)\n        return\n    reader = Reader(repo, source)\n    crop = protocol["crop"]\n    references = {}\n\n    def process(uid, role, case, attacked, oracle_image, oracle_z, oracle_score):\n        path = output / "records" / uid / role / (case["id"] + ".json")\n        if path.exists() and read(path)["status"] == "SCORED":\n            return read(path)\n        attempt = output / "attempts" / uid / role / case["id"] / (uuid.uuid4().hex + ".json")\n        write(attempt, {"status": "STARTED", "time": time.time()})\n        try:\n            predicted = old[uid, role, "predicted_h"]["h_passed_to_frozen_renderer"]\n            matrix = matrix_for(case, crop, predicted)\n            image = rectify_attacked_rgb(attacked, matrix.tolist())\n            if case["kind"] in ("oracle", "predicted"):\n                previous = old[uid, role, "oracle_h" if case["kind"] == "oracle" else "predicted_h"]\n                if not np.array_equal(np.asarray(image), np.asarray(load_saved_image(source, previous["image"]))):\n                    raise RuntimeError("Replay RGB differs from saved route; stop before residual tests")\n            z = oracle_z if case["kind"] == "oracle" else reader.observe(image)\n            scored = reader.score(z)\n            replay = replay_comparison(scored, previous) if case["kind"] in ("oracle", "predicted") else None\n            if replay is not None and not replay["passed"]:\n                write(attempt, {"status": "REPLAY_MISMATCH", "replay": replay, **scored})\n                raise RuntimeError("VAE-only reader replay mismatch; residual diagnosis halted")\n            a, b = scored["statistic"], oracle_score["statistic"]\n            result = {"status": "SCORED", "h_passed_to_renderer": matrix.tolist(), **scored,\n                      **geometry_metrics(matrix, crop), **observation_metrics(image, oracle_image, z, oracle_z),\n                      "replay": replay, "delta_score_vs_oracle": a["value"] - b["value"],\n                      "delta_registered_vs_oracle": a["registered_weighted_joint"] - b["registered_weighted_joint"],\n                      "delta_wrongmax_vs_oracle": a["wrong_key_max"] - b["wrong_key_max"],\n                      "delta_hf_vs_oracle": scored["branches"]["hf"]["registered"] - oracle_score["branches"]["hf"]["registered"],\n                      "delta_lf_vs_oracle": scored["branches"]["lf"]["registered"] - oracle_score["branches"]["lf"]["registered"],\n                      "oracle_above_reference_tau": oracle_score["above_reference_tau"],\n                      "lost_vs_oracle": oracle_score["above_reference_tau"] and not scored["above_reference_tau"]}\n            write(attempt, result)\n            write(path, result)\n            print(f"{uid[-4:]} {role} {case[\'id\']}: margin={scored[\'margin\']:.4f}", flush=True)\n            return result\n        except Exception as error:\n            result = {"status": "OPERATIONAL_ERROR", **error_data(error)}\n            # Keep detailed replay values if available, alongside the final error.\n            if attempt.exists() and read(attempt).get("status") == "REPLAY_MISMATCH":\n                result["replay_failure"] = read(attempt)\n            write(attempt, result)\n            write(path, result)\n            summarize(output)\n            raise\n\n    # Complete ALL 32 replay controls before starting any perturbation inference.\n    for uid in IDS:\n        for role in ROLES:\n            attacked = load_saved_image(source, old[uid, role, "crop_direct"]["image"])\n            oracle_image = load_saved_image(source, old[uid, role, "oracle_h"]["image"])\n            oracle_z = reader.observe(oracle_image)\n            oracle_score = reader.score(oracle_z)\n            check = replay_comparison(oracle_score, old[uid, role, "oracle_h"])\n            if not check["passed"]:\n                write(output / "replay_gate.json", {"status": "FAILED", "unit_id": uid, "role": role, "check": check})\n                raise RuntimeError("Oracle reader replay mismatch; no residual tests started")\n            references[uid, role] = (attacked, oracle_image, oracle_z, oracle_score)\n            for case in cases()[:2]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    write(output / "replay_gate.json", {"status": "PASSED", "planned": 32, "atol": ATOL, "rtol": RTOL})\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases()[2:]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    result = summarize(output)\n    print(json.dumps({"status": result["status"], "counts": result["counts"]}, indent=2), flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--source", type=Path, required=True)\n    p.add_argument("--output", type=Path, required=True)\n    p.add_argument("--repo", type=Path, required=True)\n    args = p.parse_args()\n    # Guard this even for error reporting: never create a file inside SOURCE.\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    sys.path[:0] = [str(args.repo), str(args.repo / "src")]\n    try:\n        run(args.source, args.output, args.repo)\n    except Exception as error:\n        write(args.output / "last_failure.json", error_data(error))\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
run_logged([sys.executable, str(script), '--source', str(SOURCE), '--output', str(output),
            '--repo', str(checkout)], 'residual-diagnostic', env=child_env, cwd=checkout)
summary = json.loads((output / 'summary.json').read_text())
print(json.dumps({'status':summary['status'], 'planned_scores':summary['planned_scores'],
                  'counts':summary['counts']}, indent=2))
print('Result directory:', output)


## 固定对照与解释

- 4 个对照：重放原 Oracle、重放原预测 H、Oracle 不做半像素转换、实际 H 误差固定缩小一半。
- 12 个平移点：x/y 各 ±0.5、±2、±4 像素。
- 6 个各向同性尺度点：以画布中心缩放 ±0.5%、±2%、±4%。
- 所有 H 直接作用于保存的裁剪图，**只采样一次**；不对 Oracle 输出再次变换。原预测 H 不重新估计。
- 半误差点只用于判断修正实际复合误差的作用，使用真值，不能作为盲方法的性能。
- 每张负样本走相同条件。所有点分别报告，不取最大分数，不重新调阈值，不把相关重复观测当独立负样本。
- 保存全部 LF/HF 注册／16 错钥证据、相对 Oracle 的变化、有效角点误差、RGB RMSE 和 VAE 表示差异。
  内区指标固定为 RGB `[64:448]²`、latent `[8:56]²`，不按结果选择掩膜；掩膜不参与评分。

若微小扰动就明显降低正确密钥证据，说明现有读出链对错位敏感；若较大误差仍能读取，
而预测 H 不行，应进一步检查预测复合误差和坐标机制。两种原因可以同时存在。
本轮不能单独分离 VAE 非等变性与 HF 载体相位敏感性，也不能据此宣称扩散独有机制或新方法有效。

## 恢复与输出

中断后保持 SOURCE/RUN_ID 不变运行全部单元：已完成的扰动评分跳过，重启会重读 Oracle 参照以验证读出环境。
协议、源文件哈希、脚本和环境不一致会停止，不能混用旧结果。不要同时写同一输出目录。

结果：`summary.json`、`readout.csv`、`replay_gate.json`、`protocol.json`、`environment.json`、
`records/`、`attempts/`、`execution_logs/`。输入缺失、重放图像或分数不匹配会停止；不会重新生成任何源图像。
